# Project 6: Privacy-Preserving Machine Learning
Bach Nguyen, Son Nguyen

This project explore application of the Differentially Private ID3 algorithm on the Adult Census Income dataset. 

In [9]:
import os
import os.path
import pandas as pd
import numpy as np

## Loading in the data

In [16]:
np.random.seed(42)

datadir = "data"
data = os.path.join(datadir, "adult.data")
df = pd.read_csv(data)
df.columns = ["age", "workclass", "fnlwgt", "education", "education-num", 
              "marital-status", "occupation", "relationship", "race", "sex", 
              "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]

# Dropping education-num
df.drop(columns=["education-num", "fnlwgt"], inplace=True)

In [ ]:
def age_hierarchy(min_age=17, max_age=90):
    """
    Construct generalization hierarchy for ages.

    Parameters:
        min_age: int, minimum age in dataset (default 17)
        max_age: int, maximum age in dataset (default 90)

    Returns:
        dict of {level: {age: generalized_value}}
        Levels:
            2 → 5-year bins
            3 → 10-year bins
            4 → 20-year bins
            5 → broad groups (<=24, 25-44, 45-64, >=65)
    """
    levels = {}

    # Level 2: 5-year bins
    bins_5 = {}
    for age in range(min_age, max_age + 1):
        start = min_age + 5 * ((age - min_age) // 5)
        end = min(start + 4, max_age)
        bins_5[age] = f"{start}-{end}"
    levels[2] = bins_5

    # Level 3: 10-year bins
    bins_10 = {}
    for age in range(min_age, max_age + 1):
        start = min_age + 10 * ((age - min_age) // 10)
        end = min(start + 9, max_age)
        bins_10[age] = f"{start}-{end}"
    levels[3] = bins_10

    # Level 4: 20-year bins
    bins_20 = {}
    for age in range(min_age, max_age + 1):
        start = min_age + 20 * ((age - min_age) // 20)
        end = min(start + 19, max_age)
        bins_20[age] = f"{start}-{end}"
    levels[4] = bins_20

    # Level 5: broad groups
    broad = {}
    for age in range(min_age, max_age + 1):
        if age <= 24: broad[age] = "<=24"
        elif age <= 44: broad[age] = "25-44"
        elif age <= 64: broad[age] = "45-64"
        else: broad[age] = ">=65"
    levels[5] = broad

    return levels

def generalize_age(age, level=2, min_age=17, max_age=90):
    """
    Generalize an individual age to a specified hierarchy level.

    Parameters:
        age: int, raw age value
        level: int, generalization level (1 = raw, 2–5 = bins, >=5 = Any)
        min_age: int, minimum age in dataset
        max_age: int, maximum age in dataset

    Returns:
        generalized age (str or int)
    """
    if level == 1:
        return age
    hierarchy = age_hierarchy(min_age, max_age)
    if level in hierarchy:
        return hierarchy[level][age]
    return "Any"

def hours_hierarchy(min_hours=1, max_hours=99):
    """
    Construct generalization hierarchy for hours worked per week.

    Parameters:
        min_hours: int, minimum hours in dataset (usually 1)
        max_hours: int, maximum hours in dataset (usually 99)

    Returns:
        dict of {level: {hours: generalized_value}}
        Levels:
            2 → 5-hour bins
            3 → 10-hour bins
            4 → 20-hour bins
            5 → broad groups (<=25, 26–40, 41–60, >60)
    """
    levels = {}

    # Level 2: 5-hour bins
    bins_5 = {}
    for h in range(min_hours, max_hours + 1):
        start = min_hours + 5 * ((h - min_hours) // 5)
        end = min(start + 4, max_hours)
        bins_5[h] = f"{start}-{end}"
    levels[2] = bins_5

    # Level 3: 10-hour bins
    bins_10 = {}
    for h in range(min_hours, max_hours + 1):
        start = min_hours + 10 * ((h - min_hours) // 10)
        end = min(start + 9, max_hours)
        bins_10[h] = f"{start}-{end}"
    levels[3] = bins_10

    # Level 4: 20-hour bins
    bins_20 = {}
    for h in range(min_hours, max_hours + 1):
        start = min_hours + 20 * ((h - min_hours) // 20)
        end = min(start + 19, max_hours)
        bins_20[h] = f"{start}-{end}"
    levels[4] = bins_20

    # Level 5: Broad work participation groups
    broad = {}
    for h in range(min_hours, max_hours + 1):
        if h <= 25:
            broad[h] = "<=25"
        elif h <= 40:
            broad[h] = "26-40"
        elif h <= 60:
            broad[h] = "41-60"
        else:
            broad[h] = ">60"
    levels[5] = broad

    return levels


def generalize_hours(h, level=2, min_hours=1, max_hours=99):
    """
    Generalize hours worked per week to a specified hierarchy level.

    Parameters:
        h: int, raw hours-per-week value
        level: int, generalization level (1 = raw, 2–5 = bins, >=5 = Any)
        min_hours: int
        max_hours: int

    Returns:
        generalized hours (str or int)
    """
    if level == 1:
        return h
    hierarchy = hours_hierarchy(min_hours, max_hours)
    if level in hierarchy:
        return hierarchy[level][h]
    return "Any"

## Helper Function

In [ ]:
def generalize_numeric_features(df, age_level=2, hours_level=2, min_age=17, max_age=90, min_hours=1, max_hours=99):
    """
    Apply generalization to numeric variables in the Adult dataset.

    Parameters:
        df : pandas DataFrame
        age_level : int (1-5)
        hours_level : int (1-5)
        min_age, max_age : bounds for age hierarchy
        min_hours, max_hours : bounds for hours hierarchy

    Returns:
        df with new generalized columns:
            - age_bin
            - hours_bin
            - cap_gain_bin
            - cap_loss_bin
    """

    df["age_bin"] = df["age"].apply(lambda x: generalize_age(int(x), level=age_level,
                                    min_age=min_age, max_age=max_age))

    df["hours_bin"] = df["hours-per-week"].apply(lambda x: generalize_hours(int(x), level=hours_level,
                                                min_hours=min_hours, max_hours=max_hours))

    df["cap_gain_bin"] = np.where(df["capital-gain"] > 0, "gain>0", "gain=0")
    df["cap_loss_bin"] = np.where(df["capital-loss"] > 0, "loss>0", "loss=0")

    df["age_bin"] = df["age_bin"].astype("category")
    df["hours_bin"] = df["hours_bin"].astype("category")
    df["cap_gain_bin"] = df["cap_gain_bin"].astype("category")
    df["cap_loss_bin"] = df["cap_loss_bin"].astype("category")

    return df

In [ ]:
def find_entropy_split(D, a, epsilon, label_col):
    """
    Differentially private entropy of the split on attribute `a`.

    Input:
        D (pd.DataFrame): Dataset containing feature columns and a label column.
        a (str): Name of the feature/attribute to split on.
        epsilon (float): Privacy parameter ε used for Laplace noise.
        label_col (str): Name of the label column in D.

    Output:
        float: Noisy expected entropy after splitting on attribute `a`.
    """
    n = len(D)

    tot = 0.0

    # For each unique value j of attribute a
    for j in D[a].unique():
        D_j = D[D[a] == j]
        count_D_j = len(D_j) + np.random.laplace(0.0, 1.0/epsilon)

        subtree_entropy = 0.0

        for i in D[label_col].unique():
            true_count = int(sum(D_j[label_col] == i))
            count = true_count + np.random.laplace(loc=0.0, scale=1.0/epsilon)

            pi = count / count_D_j
            subtree_entropy -= pi * np.log2(pi)

        tot += subtree_entropy * (count_D_j / n)

    return tot